# ĐỒ ÁN KHOA HỌC DỮ LIỆU: HR ANALYTICS - DỰ ĐOÁN NHÀ KHOA HỌC DỮ LIỆU ĐỔI VIỆC

**Môn học:** Khoa học Dữ liệu (CSE3040)  
**Giảng viên:** PGS. TS. Lê Chí Ngọc  
**Phương pháp luận:** CRISP-DM / KDD Process  

---
### Mục tiêu đề tài:
Xây dựng hệ thống học máy phân lớp nhị phân để dự đoán xác suất một ứng viên sau khi hoàn thành khóa đào tạo sẽ **ở lại làm việc** (`target = 0`) hay **tìm kiếm công việc mới** (`target = 1`).

In [ ]:
# 1. KHỞI TẠO CÁC THƯ VIỆN CẦN THIẾT
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score, roc_curve, auc, classification_report, confusion_matrix, ConfusionMatrixDisplay

# Cấu hình thẩm mỹ đồ thị
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

print("Đã tải thành công các thư viện!")

## 1. Khám Phá Dữ Liệu (Exploratory Data Analysis - EDA)
Đọc dữ liệu từ file `aug_train.csv` và kiểm tra cấu trúc, kiểu dữ liệu và tỷ lệ khuyết.

In [ ]:
# 2. ĐỌC DỮ LIỆU HUẤN LUYỆN
train_df = pd.read_csv("../aug_train.csv")
print("Kích thước tập dữ liệu huấn luyện:", train_df.shape)
train_df.head()

In [ ]:
# 3. KIỂM TRA TỶ LỆ DỮ LIỆU BỊ KHUYẾT (MISSING VALUES)
missing = train_df.isnull().sum()
missing_pct = (missing / len(train_df)) * 100
missing_df = pd.DataFrame({"Missing Count": missing, "Percentage (%)": missing_pct})
missing_df = missing_df[missing_df["Missing Count"] > 0].sort_values(by="Percentage (%)", ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=missing_df["Percentage (%)"], y=missing_df.index, palette="magma")
plt.title("Tỷ lệ % giá trị khuyết theo từng thuộc tính", fontsize=14, fontweight="bold")
plt.xlabel("Tỷ lệ khuyết (%)")
plt.show()

missing_df

In [ ]:
# 4. PHÂN TÍCH BIẾN MỤC TIÊU (TARGET IMBALANCE)
counts = train_df["target"].value_counts()
labels = ["Ở lại làm việc (0)", "Đổi việc (1)"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(x=labels, y=counts.values, ax=axes[0], palette=["#2b5c8f", "#d95f02"])
axes[0].set_title("Số lượng ứng viên theo nhóm mục tiêu", fontweight="bold")

axes[1].pie(counts.values, labels=labels, autopct="%1.1f%%", colors=["#2b5c8f", "#d95f02"], startangle=90, explode=[0, 0.08])
axes[1].set_title("Tỷ lệ phân bố biến mục tiêu (Mất cân bằng 3:1)", fontweight="bold")
plt.show()

In [ ]:
# 5. PHÂN TÍCH TÁC ĐỘNG CỦA CHỈ SỐ PHÁT TRIỂN THÀNH PHỐ (CITY DEVELOPMENT INDEX)
plt.figure(figsize=(10, 5))
sns.kdeplot(data=train_df, x="city_development_index", hue="target", common_norm=False, fill=True, palette=["#2b5c8f", "#d95f02"], alpha=0.4)
plt.title("Mật độ chỉ số phát triển thành phố (CDI) giữa người ở lại vs người nhảy việc", fontsize=13, fontweight="bold")
plt.xlabel("City Development Index")
plt.show()

## 2. Tiền Xử Lý Dữ Liệu & Kỹ Thuật Đặc Trưng (Feature Engineering)
Tái sử dụng module chuẩn hóa từ `src/data_preprocessing.py`:

In [ ]:
import sys
sys.path.append("..")
from src.data_preprocessing import get_train_data, get_test_data, build_column_transformer

X, y, df_clean = get_train_data("../aug_train.csv")
X_test, test_ids = get_test_data("../aug_test.csv")
print(f"X shape: {X.shape}, y shape: {y.shape}, X_test shape: {X_test.shape}")
preprocessor = build_column_transformer()
preprocessor

## 3. Huấn Luyện & So Sánh 3 Mô Hình Độc Lập
- **Mô hình 1:** Logistic Regression (L2 Regularized, class-weighted)
- **Mô hình 2:** Random Forest Classifier (150 cây, class-weighted)
- **Mô hình 3:** LightGBM Classifier (Gradient Boosting tối ưu dữ liệu bảng)

In [ ]:
from sklearn.pipeline import Pipeline

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

models = {
    "Logistic Regression": Pipeline([
        ("prep", build_column_transformer()),
        ("clf", LogisticRegression(C=0.1, class_weight="balanced", max_iter=1000, random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("prep", build_column_transformer()),
        ("clf", RandomForestClassifier(n_estimators=150, max_depth=12, min_samples_split=10, class_weight="balanced", random_state=42, n_jobs=-1))
    ]),
    "LightGBM": Pipeline([
        ("prep", build_column_transformer()),
        ("clf", LGBMClassifier(n_estimators=150, learning_rate=0.05, num_leaves=31, max_depth=6, scale_pos_weight=3.0, subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1))
    ])
}

eval_results = []
roc_curves = {}

for name, pipe in models.items():
    print(f"Đang huấn luyện {name}...")
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_val)
    y_proba = pipe.predict_proba(X_val)[:, 1]
    
    fpr, tpr, _ = roc_curve(y_val, y_proba)
    roc_score = auc(fpr, tpr)
    roc_curves[name] = (fpr, tpr, roc_score)
    
    eval_results.append({
        "Model": name,
        "ROC-AUC": round(roc_score, 4),
        "Accuracy": round((y_pred == y_val).mean(), 4)
    })

pd.DataFrame(eval_results)

In [ ]:
# 6. ĐỒ THỊ SO SÁNH ĐƯỜNG CONG ROC CURVE GIỮA CẢ 3 MÔ HÌNH
plt.figure(figsize=(9, 7))
colors = {"Logistic Regression": "#1f77b4", "Random Forest": "#2ca02c", "LightGBM": "#ff7f0e"}

for name, (fpr, tpr, score) in roc_curves.items():
    plt.plot(fpr, tpr, lw=2.5, color=colors[name], label=f"{name} (AUC = {score:.4f})")

plt.plot([0, 1], [0, 1], "k--", lw=1.5, label="Ngẫu nhiên (AUC = 0.5000)")
plt.xlabel("False Positive Rate (FPR)")
plt.ylabel("True Positive Rate (TPR / Recall)")
plt.title("So Sánh Đường Cong ROC Giữa 3 Mô Hình", fontsize=14, fontweight="bold")
plt.legend(loc="lower right", fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# 7. PHÂN TÍCH TẦM QUAN TRỌNG CỦA ĐẶC TRƯNG (FEATURE IMPORTANCE TỪ LIGHTGBM)
lgb_pipe = models["LightGBM"]
feature_names = lgb_pipe.named_steps["prep"].get_feature_names_out()
importances = lgb_pipe.named_steps["clf"].feature_importances_

feat_series = pd.Series(importances, index=feature_names).sort_values(ascending=False)
plt.figure(figsize=(10, 6))
feat_series.head(15).sort_values().plot(kind="barh", color="#ff7f0e")
plt.title("Top 15 đặc trưng chi phối mạnh nhất quyết định nhảy việc (LightGBM)", fontsize=13, fontweight="bold")
plt.xlabel("Importance Score")
plt.show()

In [ ]:
# 8. XUẤT FILE KẾT QUẢ DỰ ĐOÁN KIỂM THỬ CHO TẬP TEST (SUBMISSION)
best_model = models["LightGBM"]
test_preds = best_model.predict_proba(X_test)[:, 1]

sub_df = pd.DataFrame({
    "enrollee_id": test_ids,
    "target": np.round(test_preds, 4)
})
sub_df.to_csv("../submissions/submission_notebook_lightgbm.csv", index=False)
print(f"Đã xuất kết quả kiểm thử cho {len(sub_df)} ứng viên!")
sub_df.head()